# Notebook 02 — Prepare a Candidate Train/Validation/Test Manifest

**Goal:** Create a reproducible, **provisional** 3-way split for the Pakistani vehicle dataset while preserving the publisher's test partition.

The publisher's 3,096 `train` images are divided into train/validation at a proposed validation fraction of **0.15**. Its **751 test images remain assigned to test**. No images are moved, removed, cropped, changed, or copied.

> **Important:** This notebook creates a **candidate** manifest, not the final approved split. It also flags perceptually similar images across boundaries, which require review. No test results or model measurements are generated.

Smallest observed original training class: 13 images; largest: 270. Minority-class validation estimates will be noisy.

## 1. What is stratification?

If we randomly select validation photos from the entire training folder, some rare vehicle models might have none left for validation.

A **stratified split** keeps each class represented approximately in proportion to its original size. For a class with 13 training images, 15% is about 2 validation images and 11 training images. This maintains class coverage but does **not** create additional data.

The test set remains separate for **final evaluation only**. We may inspect file metadata for leakage auditing, but we must not use test accuracy to select a model or augmentations.

In [1]:
from pathlib import Path
from collections import Counter
from hashlib import sha256
from itertools import product
import json

import pandas as pd
import yaml
from PIL import Image
from sklearn.model_selection import train_test_split

HERE = Path.cwd().resolve()
REPO = next((p for p in (HERE, *HERE.parents) if (p / "configs" / "experiments.yaml").exists()), None)
if REPO is None:
    raise RuntimeError("Open the notebook within the project repository.")

with (REPO / "configs" / "experiments.yaml").open(encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

D = CFG["datasets"]["pakistan"]
SEED = int(CFG["random_seed"])
VAL_FRACTION = float(D["split"]["validation_fraction_of_publisher_train"])
DATA_ROOT = REPO / D["data_dir"]
OUT = REPO / D["split_manifest_dir"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

assert 0 < VAL_FRACTION < 1
print("Repository:", REPO)
print("Dataset root:", DATA_ROOT)
print("Split policy: original test untouched; validation fraction of publisher training =", VAL_FRACTION)
print("Seed:", SEED)

Repository: /Users/bata/Documents/vehicle-make-model-recognition
Dataset root: /Users/bata/Documents/vehicle-make-model-recognition/data/pakistan
Split policy: original test untouched; validation fraction of publisher training = 0.15
Seed: 42


## 2. Find the publisher's train and test directories

We require **exactly one pair** of matching `train` and `test` folders below `data/pakistan/`. This prevents accidentally mixing the split dataset with an unsplit duplicate image tree.

Images are referenced by paths relative to the repository, so the same manifest works on every team member's machine after downloading the dataset into the same folder.

In [2]:
pairs = []
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Dataset directory missing: {DATA_ROOT}")
for p in DATA_ROOT.rglob("*"):
    if p.is_dir() and p.name.lower() == "train":
        siblings = [x for x in p.parent.iterdir() if x.is_dir() and x.name.lower() == "test"]
        if len(siblings) == 1:
            pairs.append((p, siblings[0]))
if len(pairs) != 1:
    raise RuntimeError(f"Expected exactly one publisher train/test folder pair; found {len(pairs)}.")
TRAIN_ROOT, TEST_ROOT = pairs[0]
print("Publisher training folder:", TRAIN_ROOT.relative_to(REPO))
print("Publisher test folder:", TEST_ROOT.relative_to(REPO))

def collect_images(folder, publisher_split):
    rows = []
    class_dirs = sorted((p for p in folder.iterdir() if p.is_dir()), key=lambda p: p.name)
    for class_dir in class_dirs:
        for path in sorted(class_dir.rglob("*"), key=lambda p: p.as_posix()):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({
                    "relative_path": path.relative_to(REPO).as_posix(),
                    "class_name": class_dir.name,
                    "publisher_split": publisher_split,
                })
    orphan_images = [
        p.name for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]
    if orphan_images:
        raise RuntimeError(f"Images found directly within {folder}: {orphan_images[:5]}")
    return rows

orig_train = collect_images(TRAIN_ROOT, "train")
orig_test = collect_images(TEST_ROOT, "test")
train_labels = sorted({r["class_name"] for r in orig_train})
test_labels = sorted({r["class_name"] for r in orig_test})
assert train_labels == test_labels, "Publisher train/test class sets differ; investigate."
assert len(orig_train) == D["audited_publisher_split_counts"]["train"]
assert len(orig_test) == D["audited_publisher_split_counts"]["test"]
assert len(orig_train) + len(orig_test) == D["verified_images"]
assert len(train_labels) == D["verified_classes"]
assert len(set(x["relative_path"] for x in orig_train + orig_test)) == len(orig_train + orig_test)

counts = Counter(r["class_name"] for r in orig_train)
print(f"Train images: {len(orig_train):,}; Test images: {len(orig_test):,}")
print(f"Classes: {len(train_labels)}; Min original train/class: {min(counts.values())}; Max: {max(counts.values())}")
print("All classes exist in both publisher splits.")

Publisher training folder: data/pakistan/Splited_dataset/train
Publisher test folder: data/pakistan/Splited_dataset/test
Train images: 3,096; Test images: 751
Classes: 48; Min original train/class: 13; Max: 270
All classes exist in both publisher splits.


## 3. Create the same candidate split on every computer

`train_test_split` with a fixed seed and `stratify` yields repeatable assignments, provided the files and the sklearn version are the same. We also save an explicit manifest so every model uses **exactly the same** split even if libraries differ across computers.

**Note:** Different frames of the same vehicle may not be independent. We will check candidate perceptual similarities next. If actual source-video or vehicle-group IDs become available, a group-aware split is preferable.

In [3]:
train_rows, val_rows = train_test_split(
    orig_train,
    test_size=VAL_FRACTION,
    random_state=SEED,
    shuffle=True,
    stratify=[r["class_name"] for r in orig_train],
)
CLASS_TO_INDEX = {name: i for i, name in enumerate(train_labels)}
all_rows = (
    [{**r, "split": "train"} for r in train_rows]
    + [{**r, "split": "val"} for r in val_rows]
    + [{**r, "split": "test"} for r in orig_test]
)
manifest = pd.DataFrame(all_rows)
manifest["class_idx"] = manifest["class_name"].map(CLASS_TO_INDEX).astype(int)
manifest = manifest[["relative_path", "split", "publisher_split", "class_name", "class_idx"]]
manifest = manifest.sort_values(["split", "class_idx", "relative_path"]).reset_index(drop=True)

assert len(manifest) == 3847
assert not manifest["relative_path"].duplicated().any()
assert (manifest.loc[manifest["split"].eq("test"), "publisher_split"] == "test").all()
assert (manifest.loc[manifest["split"].ne("test"), "publisher_split"] == "train").all()
assert manifest.groupby("split")["class_name"].nunique().to_dict() == {"test": 48, "train": 48, "val": 48}

table = pd.crosstab(manifest["class_name"], manifest["split"])
print("Candidate subset sizes:", manifest["split"].value_counts().to_dict())
print("Minimum examples/class:", table.min().to_dict())
display(table.sort_values("train").head(12))

Candidate subset sizes: {'train': 2631, 'test': 751, 'val': 465}
Minimum examples/class: {'test': 3, 'train': 11, 'val': 2}


split,test,train,val
class_name,,,
Suzuki_carry,3,11,2
Toyota_Landcruser,4,14,3
Suzuki_margala,4,14,2
Toyota_premio,4,15,3
Honda_civic_1994,4,15,3
Toyota_Prado,5,17,3
Toyota_axio,4,17,3
Honda_Grace,5,18,3
Suzuki_Every,5,18,3


## 4. Screen for perceptual near-duplicate candidates

Two adjacent frames of the same car may have **different SHA-256 file hashes**, yet look nearly identical. We compute a small **difference hash (dHash)** from each image: each hash is a 64-bit summary of adjacent-pixel brightness comparisons.

Small **Hamming distance** (different bits between hashes) indicates potentially similar appearance. The threshold of `3` is a *screening heuristic* and is **not a validated duplicate detector**. It may miss similar frames and flag unrelated cars. The presence of flagged candidates needs visual review.

Two comparisons:
- Publisher `train` vs publisher `test`: check possible original test contamination.
- Proposed `train` vs proposed `val`: check our newly generated holdout.

Do not automatically delete, relabel or move images based on dHash alone.

In [4]:
def dhash(path: Path) -> int:
    with Image.open(path) as im:
        pixels = list(im.convert("L").resize((9, 8), Image.Resampling.LANCZOS).getdata())
    result = 0
    for y in range(8):
        for x in range(8):
            result |= int(pixels[y * 9 + x] > pixels[y * 9 + x + 1]) << (y * 8 + x)
    return result

all_paths = sorted(manifest["relative_path"].unique())
hashes = {relative: dhash(REPO / relative) for relative in all_paths}
print("Perceptual hashes generated:", len(hashes))

def find_close_pairs(first, second, pair_type, threshold=3, example_limit=100):
    candidate_count, samples = 0, []
    for left in first:
        h = hashes[left["relative_path"]]
        for right in second:
            distance = (h ^ hashes[right["relative_path"]]).bit_count()
            if distance <= threshold:
                candidate_count += 1
                if len(samples) < example_limit:
                    samples.append({
                        "boundary": pair_type,
                        "distance": distance,
                        "left_path": left["relative_path"],
                        "right_path": right["relative_path"],
                        "same_class": left["class_name"] == right["class_name"],
                    })
    return candidate_count, samples

original_count, original_examples = find_close_pairs(orig_train, orig_test, "publisher_train_vs_test")
candidate_count, candidate_examples = find_close_pairs(train_rows, val_rows, "candidate_train_vs_val")
near_examples = pd.DataFrame(original_examples + candidate_examples,
    columns=["boundary", "distance", "left_path", "right_path", "same_class"])
print("Near-similar hash PAIRS across publisher train/test:", original_count)
print("Near-similar hash PAIRS across candidate train/validation:", candidate_count)
print("Showing at most 100 candidates per comparison; counts include all threshold matches.")
if len(near_examples):
    display(near_examples.sort_values(["distance", "boundary"]).head(25))
else:
    print("No dHash candidates found at threshold 3. This does NOT prove absence of near-duplicates.")

/var/folders/qx/t5660_gn1639zk2tzf5bgdbw0000gn/T/ipykernel_59002/3493223393.py:3: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(im.convert("L").resize((9, 8), Image.Resampling.LANCZOS).getdata())


Perceptual hashes generated: 3847
Near-similar hash PAIRS across publisher train/test: 23
Near-similar hash PAIRS across candidate train/validation: 12
Showing at most 100 candidates per comparison; counts include all threshold matches.


,boundary,distance,left_path,right_path,same_class
23,candidate_train_vs_val,0,data/pakistan/Splited_dataset/train/Honda_city...,data/pakistan/Splited_dataset/train/Honda_city...,True
25,candidate_train_vs_val,0,data/pakistan/Splited_dataset/train/Suzuki_swi...,data/pakistan/Splited_dataset/train/Suzuki_swi...,True
26,candidate_train_vs_val,0,data/pakistan/Splited_dataset/train/Toyota_axi...,data/pakistan/Splited_dataset/train/Toyota_axi...,True
27,candidate_train_vs_val,0,data/pakistan/Splited_dataset/train/Suzuki_cul...,data/pakistan/Splited_dataset/train/Suzuki_cul...,True
28,candidate_train_vs_val,0,data/pakistan/Splited_dataset/train/Daiatsu_Hi...,data/pakistan/Splited_dataset/train/Daiatsu_Hi...,True
29,candidate_train_vs_val,0,data/pakistan/Splited_dataset/train/Toyota_Vit...,data/pakistan/Splited_dataset/train/Toyota_Vit...,True
31,candidate_train_vs_val,0,data/pakistan/Splited_dataset/train/Daiatsu_Co...,data/pakistan/Splited_dataset/train/Daiatsu_Co...,True
32,candidate_train_vs_val,0,data/pakistan/Splited_dataset/train/Suzuki_cul...,data/pakistan/Splited_dataset/train/Suzuki_cul...,True
33,candidate_train_vs_val,0,data/pakistan/Splited_dataset/train/Suzuki_swi...,data/pakistan/Splited_dataset/train/Suzuki_swi...,True
34,candidate_train_vs_val,0,data/pakistan/Splited_dataset/train/Honda_City...,data/pakistan/Splited_dataset/train/Honda_City...,True


## 5. Save provisional, reproducible artifacts

These outputs are **candidates**, not an approved split:

- `splits/pakistan/candidate_manifest.csv`: path, subset, publisher subset, class name/index.
- `splits/pakistan/candidate_class_to_idx.json`: frozen 0–47 class mapping.
- `splits/pakistan/candidate_near_duplicate_pairs.csv`: up to 100 sampled near-similar image pairs from each boundary.
- `splits/pakistan/candidate_split_metadata.json`: record counts, seed, fraction, near-similar pair counts and SHA-256 checksum.

No files under `data/` are edited. **Do not train from this candidate manifest until the team reviews possible leakage and explicitly promotes an approved version.**

In [5]:
OUT.mkdir(parents=True, exist_ok=True)
manifest_path = OUT / "candidate_manifest.csv"
manifest.to_csv(manifest_path, index=False)
(OUT / "candidate_class_to_idx.json").write_text(
    json.dumps(CLASS_TO_INDEX, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
near_examples.to_csv(OUT / "candidate_near_duplicate_pairs.csv", index=False)
digest = sha256(manifest_path.read_bytes()).hexdigest()
metadata = {
    "status": "CANDIDATE_NOT_APPROVED",
    "source": "Mendeley 2022 dataset / publisher train+test",
    "seed": SEED,
    "validation_fraction_of_publisher_train": VAL_FRACTION,
    "counts": {k: int(v) for k, v in manifest["split"].value_counts().to_dict().items()},
    "class_count": len(CLASS_TO_INDEX),
    "manifest_sha256": digest,
    "dhash_hamming_threshold": 3,
    "publisher_train_vs_test_candidate_pairs": original_count,
    "candidate_train_vs_val_candidate_pairs": candidate_count,
    "limitation": "dHash may miss visually near-identical frames; manual inspection/group metadata remain necessary",
}
(OUT / "candidate_split_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print("Candidate manifest written:", manifest_path)
print("Manifest SHA-256:", digest)
print("Files saved under:", OUT)
print("STATUS: CANDIDATE, requires near-duplicate review before approval.")
print("No images copied, moved, modified or deleted.")

Candidate manifest written: /Users/bata/Documents/vehicle-make-model-recognition/splits/pakistan/candidate_manifest.csv
Manifest SHA-256: 97fa106c50e35fd184ea6ee8535e345599555f9af50c7ee6e2e3863b5da1ca82
Files saved under: /Users/bata/Documents/vehicle-make-model-recognition/splits/pakistan
STATUS: CANDIDATE, requires near-duplicate review before approval.
No images copied, moved, modified or deleted.


## Checkpoint for Project HQ

Share the printed results from **Sections 3 and 4**:
1. Candidate train/validation/test sizes and per-split minimum images per class.
2. Count of dHash near-similar **pairs** across publisher train/test.
3. Count of dHash near-similar **pairs** across candidate train/validation.
4. Whether any classes have zero images in any subset (the notebook asserts against this).

The counts above are not model accuracy measurements. Later, after reviewing candidates, we will approve the fixed manifest for both Xception and ConvNeXt. We will **not** change the publisher test allocation or use test accuracy to select a training configuration.